In [0]:
# Install select libraries that depend on databricks sdk >= 0.65.0

# Install a coherent 0.3.x stack (compatible with unitycatalog-langchain and databricks-langchain)
%pip install -U \
  "langchain<0.4,>=0.3.27" \
  "langchain-core<0.4,>=0.3.79" \
  "langchain-community<0.4,>=0.2" \
  "langchain-text-splitters<1.0,>=0.3.9" \
  "langchain-openai<0.3,>=0.2.0" \
  "pydantic>=2.0.0,<3.0.0" \
  "databricks-sdk>=0.65.0" \
  "databricks-langchain==0.8.2"

# Restart the Python VM so the environment picks up the new packages
%restart_python

In [0]:
%run ../../Includes/_common

In [0]:
# Create a python DA object from the dbacademy.ops.meta table
DA = DBAcademyHelper()
DA.init()

In [0]:
def use_uc_env():
    catalog_name = DA.catalog_name
    schema_name = DA.schema_name 

    spark.sql(f"USE CATALOG {DA.catalog_name}")
    spark.sql(f"USE SCHEMA {DA.schema_name}")
    return catalog_name, schema_name 

In [0]:
def process_airbnb_dataset(databricks_share_name: str):
    # Read the CSV file from the volume with headers
    df = spark.read.format("csv") \
        .option("header", "true") \
        .option("inferSchema", "true") \
        .option("multiLine", "true") \
        .option("escape", '"') \
        .load(f"/Volumes/{databricks_share_name}/v01/sf-listings/sf-airbnb.csv")

    # Write as a Delta table
    df.write.format("delta") \
        .mode("overwrite") \
        .saveAsTable("sf_airbnb_listings")
        
    print(f"✅ Successfully created table {DA.catalog_name}.{DA.schema_name}.sf_airbnb_listings.")

In [0]:
# Set UC environment
catalog_name, schema_name = use_uc_env()

# Process the Airbnb dataset
process_airbnb_dataset(
        databricks_share_name = "dbacademy_airbnb"
        )

In [0]:
def set_environment_and_tools(catalog_name:str, schema_name:str) -> None:
    """
    Sets the environment and tools for the notebook.
    """

    query1 = f"""
    USE CATALOG {catalog_name}
    """
    query2 = f"""
    USE SCHEMA {schema_name}
    """
    query3 = """
    DROP FUNCTION IF EXISTS avg_neigh_price
    """
    query4 = """
    DROP FUNCTION IF EXISTS cnt_by_room_type
    """
    query5 = f"""
    CREATE OR REPLACE FUNCTION avg_neigh_price(
    neighborhood_name STRING COMMENT "The neighborhood name to filter by (e.g., 'Mission', 'Upper Market')"
    )
    RETURNS DOUBLE
    LANGUAGE SQL
    DETERMINISTIC
    COMMENT 'Calculates the average listing price for a specific neighborhood in San Francisco. Returns the average price as a numeric value. Price strings are cleaned and converted to numeric values before averaging.'
    RETURN 
    SELECT AVG(CAST(REGEXP_REPLACE(price, '[^0-9.]', '') AS DOUBLE))
    FROM sf_airbnb_listings
    WHERE neighbourhood_cleansed = neighborhood_name
    AND price IS NOT NULL
    AND REGEXP_REPLACE(price, '[^0-9.]', '') != ''
    """
    query6 = f"""
    CREATE OR REPLACE FUNCTION cnt_by_room_type(
    neighborhood_name STRING COMMENT "The neighborhood name to filter by",
    room_type_filter STRING COMMENT "The room type to count (e.g., 'Private room' or 'Shared room')"
    )
    RETURNS BIGINT
    LANGUAGE SQL
    DETERMINISTIC
    COMMENT 'Counts the number of Airbnb listings for a specific room type in a given neighborhood. Returns the count as an integer.'
    RETURN
    SELECT COUNT(*)
    FROM sf_airbnb_listings
    WHERE neighbourhood_cleansed = neighborhood_name
        AND room_type = room_type_filter
    """
    print(f"Using catalog `{catalog_name}` and schema `{schema_name}`")
    spark.sql(query1)
    spark.sql(query2)
    
    print("Creating functions...")
    spark.sql(query3).collect()
    spark.sql(query4).collect()
    
    spark.sql(query5).collect()
    print(f"Created function avg_price_by_neighborhood")
    
    spark.sql(query6).collect()
    print(f"Created function cnt_by_room_type")
    return None

In [0]:
set_environment_and_tools(catalog_name, schema_name)